# Notebook 06 — Diễn giải mô hình
## Feature Importance + SHAP Values

**Mục tiêu:**
1. Feature importance từ RandomForest/XGBoost
2. SHAP summary plot — tác động toàn bộ dataset
3. SHAP bar plot — feature importance dạng |SHAP|
4. SHAP waterfall — giải thích từng dự đoán


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

from src.visualization.plots import plot_feature_importance
from src.data.load_data import FEATURE_COLS

plt.rcParams['figure.dpi'] = 120
shap.initjs()  # JavaScript cho SHAP plots trong Jupyter
print('Import thành công ✅')

## 1. Load model và data

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze().values.astype(int)

best_model = joblib.load('../models/best_model.pkl')
model_name = type(best_model).__name__

print(f'Model: {model_name}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

## 2. Feature Importance (tree-based)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    fig = plot_feature_importance(
        best_model, FEATURE_COLS, top_n=15,
        model_name=model_name,
        save_path='../reports/figures/06_feature_importance.png'
    )
    plt.show()

    # Top 5 features
    importances = best_model.feature_importances_
    top5_idx = np.argsort(importances)[::-1][:5]
    print('\nTop 5 yếu tố quan trọng nhất:')
    for rank, idx in enumerate(top5_idx, 1):
        print(f'  {rank}. {FEATURE_COLS[idx]}: {importances[idx]:.4f}')
else:
    print(f'Mô hình {model_name} không có feature_importances_')
    print('Hãy dùng XGBoost hoặc Random Forest cho feature importance')

## 3. SHAP Analysis

In [ ]:
# Dùng sample nhỏ để SHAP nhanh hơn
X_shap = X_test.sample(min(200, len(X_test)), random_state=42)
print(f'Tính SHAP trên {len(X_shap)} mẫu...')

# Chọn explainer phù hợp
try:
    from xgboost import XGBClassifier
    if isinstance(best_model, XGBClassifier):
        explainer = shap.TreeExplainer(best_model)
    else:
        explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)
    print('TreeExplainer SHAP đã tính xong ✅')
except Exception as e:
    print(f'TreeExplainer thất bại: {e}')
    print('Dùng KernelExplainer (chậm hơn)...')
    X_bg = shap.sample(X_train, 50)
    explainer = shap.KernelExplainer(best_model.predict_proba, X_bg)
    shap_values = explainer.shap_values(X_shap, nsamples=100)
    print('KernelExplainer SHAP đã tính xong ✅')

In [ ]:
# SHAP Summary Plot (cho lớp Dropout = class 0)
print('SHAP Summary Plot — Lớp Dropout:')
if isinstance(shap_values, list):
    shap_dropout = shap_values[0]  # class 0 = Dropout
else:
    shap_dropout = shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_dropout, X_shap,
    feature_names=FEATURE_COLS,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Summary Plot — Lớp Dropout', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_shap_summary_dropout.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot — Feature importance tổng hợp
plt.figure(figsize=(9, 6))
shap.summary_plot(
    shap_dropout, X_shap,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP Bar Plot — Mean |SHAP| value (Dropout)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_shap_bar_dropout.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Waterfall — Giải thích 1 sinh viên cụ thể (mẫu có P(Dropout) cao nhất)
y_prob_test = best_model.predict_proba(X_test)
high_risk_idx = np.argmax(y_prob_test[:, 0])  # Mẫu có P(Dropout) cao nhất

X_sample = X_test.iloc[[high_risk_idx]]
shap_sample = explainer.shap_values(X_sample)
if isinstance(shap_sample, list):
    shap_sample_dropout = shap_sample[0][0]
else:
    shap_sample_dropout = shap_sample[0]

plt.figure(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_sample_dropout,
        base_values=explainer.expected_value[0] if isinstance(explainer.expected_value, list) else explainer.expected_value,
        data=X_sample.iloc[0].values,
        feature_names=FEATURE_COLS,
    ),
    max_display=15,
    show=False
)
plt.title(f'SHAP Waterfall — Sinh viên nguy cơ cao (idx={high_risk_idx})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSinh viên nguy cơ cao nhất (idx={high_risk_idx}):')
print(f'  P(Dropout) = {y_prob_test[high_risk_idx, 0]:.3f}')
print(f'  Nhãn thực  = {["Dropout","Enrolled","Graduate"][y_test[high_risk_idx]]}')

## 4. Top yếu tố nguy cơ — Tổng kết

In [ ]:
# Tổng hợp từ SHAP
mean_abs_shap = np.abs(shap_dropout).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Mean_SHAP': mean_abs_shap
}).sort_values('Mean_SHAP', ascending=False)

print('=== TOP 10 YẾU TỐ NGUY CƠ (SHAP) ===')
for i, row in shap_importance.head(10).iterrows():
    print(f'  {shap_importance.index.get_loc(i)+1:2d}. {row["Feature"]:<45} SHAP={row["Mean_SHAP"]:.4f}')

print()
print('Giải thích:')
print('  • SHAP value dương → tăng xác suất Dropout')
print('  • SHAP value âm   → giảm xác suất Dropout')
print('  • |SHAP| càng lớn → yếu tố càng quan trọng')